In [17]:
import sys
import os
from datetime import datetime, timezone
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)

from common.const import (
    LOCAL_CATEGORIES_FILE_PATH, BATCH_SIZE,
    SCHEMA, TABLE_RAW_PRODUCTS, TABLE_RAW_REVIEWS
)
from common.helper import load_hf_data, connect_postgresql, parse_price
import json
from tqdm import tqdm
from psycopg2.extras import execute_values

with open(LOCAL_CATEGORIES_FILE_PATH, 'r') as f:
    for line in f:
        categories = sorted([line.strip() for line in f])

categories = ["All_Beauty"] # This line is for testing only (selecting a subset of categories).

# Create schema and tables

In [ ]:
sql = f"""
CREATE SCHEMA IF NOT EXISTS {SCHEMA};

DROP TABLE IF EXISTS {SCHEMA}.{TABLE_RAW_REVIEWS};

CREATE TABLE IF NOT EXISTS {SCHEMA}.{TABLE_RAW_REVIEWS} (
    id BIGSERIAL PRIMARY KEY,

    timestamp TIMESTAMPTZ NOT NULL,

    user_id VARCHAR(100) NOT NULL,
    parent_asin VARCHAR(100) NOT NULL,
    asin VARCHAR(100) NOT NULL,

    rating SMALLINT CHECK (rating BETWEEN 1 AND 5),

    title TEXT,
    text TEXT,

    images JSONB,
    helpful_vote INT DEFAULT 0,
    verified_purchase BOOLEAN,

    product_category VARCHAR(100)
);
CREATE INDEX idx_reviews_product_time
ON {SCHEMA}.{TABLE_RAW_REVIEWS} (parent_asin, timestamp DESC);

CREATE INDEX idx_reviews_user_time
ON {SCHEMA}.{TABLE_RAW_REVIEWS} (user_id, timestamp DESC);

DROP TABLE IF EXISTS {SCHEMA}.{TABLE_RAW_PRODUCTS};

CREATE TABLE IF NOT EXISTS {SCHEMA}.{TABLE_RAW_PRODUCTS} (
    parent_asin VARCHAR(100) PRIMARY KEY,

    main_category VARCHAR(50),
    title TEXT NOT NULL,

    average_rating REAL,
    rating_number INT,

    features TEXT[],
    description TEXT[],

    price NUMERIC(10, 2),

    images JSONB,
    videos JSONB,

    store TEXT,
    categories TEXT[],
    details JSONB,

    bought_together TEXT,
    subtitle TEXT,
    author TEXT
);
"""
# CREATE INDEX idx_products_search
# ON products USING GIN (search_vector); 
# already exceed storage so not not create index for now.

conn = connect_postgresql()
cursor = conn.cursor()

cursor.execute(sql)
conn.commit()
conn.close()

# Import data to `raw_products` table

In [30]:
conn = connect_postgresql()
cursor = conn.cursor()

insert_query = f"""
INSERT INTO {SCHEMA}.{TABLE_RAW_PRODUCTS} (
    main_category,
    title,
    average_rating,
    rating_number,
    features,
    description,
    price,
    images,
    videos,
    store,
    categories,
    details,
    parent_asin,
    bought_together,
    subtitle,
    author
)
VALUES %s
"""

batch = []

for category in categories[:1]:
    data = load_hf_data(
        root_split="raw",
        data_type="meta",
        category=category,
    )
    for record in tqdm(data):
        batch.append((
            record["main_category"],
            record["title"],
            record["average_rating"],
            record["rating_number"],
            record["features"],
            record["description"],
            parse_price(record["price"]),
            json.dumps(record["images"]),
            json.dumps(record["videos"]),
            record["store"],
            record["categories"],
            json.dumps(json.loads(record["details"])),
            record["parent_asin"],
            record["bought_together"],
            record["subtitle"],
            record["author"]
        ))

        if len(batch) >= BATCH_SIZE:
            execute_values(cursor, insert_query, batch)
            conn.commit()
            batch.clear()
    #         break # Uncommenting this and the line below for debugging
    # break

    # Insert remaining rows
    if batch:
        execute_values(cursor, insert_query, batch)
        conn.commit()
        
conn.close()

112590it [02:37, 714.44it/s] 


# Import data to `raw_reviews` table

In [31]:
conn = connect_postgresql()
cursor = conn.cursor()

insert_query = f"""
INSERT INTO {SCHEMA}.{TABLE_RAW_REVIEWS} (
    timestamp,
    user_id,
    parent_asin,
    asin,
    rating,
    title,
    text,
    images,
    helpful_vote,
    verified_purchase,
    product_category
)
VALUES %s
"""

batch = []

for category in categories:
    data = load_hf_data(
        root_split="raw",
        data_type="review",
        category=category,
    )

    for record in tqdm(data):
        batch.append((
            datetime.fromtimestamp(record["timestamp"] / 1000, tz=timezone.utc),
            record["user_id"],
            record["parent_asin"],
            record["asin"],
            record["rating"],
            record["title"],
            record["text"],
            json.dumps(record["images"]),
            record["helpful_vote"],
            record["verified_purchase"],
            category
        ))

        if len(batch) >= BATCH_SIZE:
            execute_values(cursor, insert_query, batch)
            conn.commit()
            batch.clear()
    #         break # Uncommenting this and the line below for debugging
    # break

    # Insert remaining rows
    if batch:
        execute_values(cursor, insert_query, batch)
        conn.commit()
        
conn.close()

701528it [08:43, 1339.70it/s]


# Measure query performance

## Query 1: Given a product ID (parent_asin), retrieve the product title and full product description.

In [32]:
import time
import numpy as np
from common.helper import measure_postgres_query_time

num_iter = 10
conn = connect_postgresql()

# with conn.cursor() as cur:
#     cur.execute(f"""
#         SELECT parent_asin
#         FROM {SCHEMA}.{TABLE_RAW_PRODUCTS}
#         ORDER BY RANDOM()
#         LIMIT %s
#     """, (num_iter,))
#     random_list = cur.fetchall()

# print(random_list)

query_duration_logs = []

for parent_asin in ['B07484MS4G', 'B01FYDF3VI', 'B01C3PMM7S', 'B012HUTKTM', 'B075R6NQ1W', 'B08DCZGQJ5', 'B077WBYPLG', 'B001KYU8UM', 'B0919HCJDN', 'B08MQCXGJM']:
    query = f"""
        SELECT title, description
        FROM {SCHEMA}.{TABLE_RAW_PRODUCTS}
        WHERE parent_asin = %s
    """
    execution_time = measure_postgres_query_time(conn, query, (parent_asin,))
    query_duration_logs.append(execution_time) # cuz planning time can easil

print(f"Over {num_iter} runs:")
print(f"Median query duration: {np.median(query_duration_logs):.3f} ms")
print(f"Mean query duration: {np.mean(query_duration_logs):.3f} ms")

conn.close()


Over 10 runs:
Median query duration: 1.987 ms
Mean query duration: 5.460 ms


## Recent Reviews: Given a product ID (parent asin), retrieve the most recent N customer reviews, including rating, review title, review text, and timestamp.

In [33]:
num_iter = 10
N = 10
conn = connect_postgresql()

# random products
# with conn.cursor() as cur:
#     cur.execute(f"""
#         SELECT parent_asin
#         FROM {SCHEMA}.{TABLE_RAW_PRODUCTS}
#         ORDER BY RANDOM()
#         LIMIT %s
#     """, (num_iter,))
#     random_list = cur.fetchall()

durations = []

for parent_asin in ['B07484MS4G', 'B01FYDF3VI', 'B01C3PMM7S', 'B012HUTKTM', 'B075R6NQ1W', 'B08DCZGQJ5', 'B077WBYPLG', 'B001KYU8UM', 'B0919HCJDN', 'B08MQCXGJM']:
    query = f"""
        SELECT rating, title, text, timestamp
        FROM {SCHEMA}.{TABLE_RAW_REVIEWS}
        WHERE parent_asin = %s
        ORDER BY timestamp DESC
        LIMIT %s
    """
    duration = measure_postgres_query_time(
        conn, query, (parent_asin, N)
    )
    durations.append(duration)

print("Recent Reviews Query:")
print(f"Median: {np.median(durations):.3f} ms")
print(f"Mean:   {np.mean(durations):.3f} ms")


Recent Reviews Query:
Median: 4.635 ms
Mean:   5.675 ms


## Keyword Search: Given a keyword, return all products whose title or description contains the keyword.

In [34]:
num_iter = 10
keywords = ['Plumping Whip', 'Eye Sticker', 'Hair Scrunchies Colorful', 'facial jade roller', 'Professional Supreme Performance Braiding', 'vintage English floral fragrance', 'Water Weightless Hydration Conditioner',
'hottest beauty product of 2019', 'Mist Coolant Lubrication Spray System', 'keep the skin calm and soothed']

conn = connect_postgresql()

durations = []

query = f"""
    SELECT parent_asin, title
    FROM {SCHEMA}.{TABLE_RAW_PRODUCTS}
    WHERE
        title ILIKE CONCAT('%%', %s, '%%')
        OR array_to_string(description, ' ') ILIKE CONCAT('%%', %s, '%%')
"""

for keyword in keywords:
    for _ in range(num_iter):
        duration = measure_postgres_query_time(
            conn, query, (keyword, keyword)
        )
        durations.append(duration)

print("Keyword Search Query (Multiple Keywords):")
print(f"Median: {np.median(durations):.3f} ms")
print(f"Mean:   {np.mean(durations):.3f} ms")

Keyword Search Query (Multiple Keywords):
Median: 667.564 ms
Mean:   688.043 ms


## User Review History: Given a user ID, retrieve all products reviewed by that user along with the corresponding ratings and review dates.


In [35]:
num_iter = 10
conn = connect_postgresql()

# random users
# with conn.cursor() as cur:
#     cur.execute(f"""
#         SELECT user_id
#         FROM (
#             SELECT DISTINCT user_id
#             FROM {SCHEMA}.{TABLE_RAW_REVIEWS}
#         ) t
#         ORDER BY RANDOM()
#         LIMIT %s
#     """, (num_iter,))
#     random_users = cur.fetchall()
random_users = ['AG5JJZKUDTC5PREHPTI5NOC75M2Q', 'AFQIHDRZZ7RGRLSFKDGCYPBHHTKA', 'AFXCPSCWLOSVQZHN7FAECOYAT3RA', 
                'AHT6VS3G3JSA7KBOCBT6KPUCJX7A', 'AH4G4SNKSIWZNGZ5EDNJTRORKIMQ', 'AG4JLUSDCYTYSSE5SQADXOPATLTA', 
                'AGXVAAK6XJ7C2GUTYQEMBOGQ7JGQ', 'AGKPEXI6SAACYVXRNMICW3NKG2XA', 'AHA272AOVKWZUXL672G653A2MTIA', 
                'AH6X5VUV5AUGO3G55G7AL6GZ2QZQ']
durations = []

for user_id in random_users:
    query = f"""
        SELECT
            r.parent_asin,
            p.title,
            r.rating,
            r.timestamp
        FROM {SCHEMA}.{TABLE_RAW_REVIEWS} r
        JOIN {SCHEMA}.{TABLE_RAW_PRODUCTS} p
            ON p.parent_asin = r.parent_asin
        WHERE r.user_id = %s
        ORDER BY r.timestamp DESC
    """
    duration = measure_postgres_query_time(
        conn, query, (user_id,)
    )
    durations.append(duration)

print("User Review History Query:")
print(f"Median: {np.median(durations):.3f} ms")
print(f"Mean:   {np.mean(durations):.3f} ms")

User Review History Query:
Median: 4.931 ms
Mean:   8.680 ms


## Product Statistics: Retrieve the average rating, rating distribution (count per star level), and total number of reviews for a given product.

In [36]:
num_iter = 10
conn = connect_postgresql()



durations = []

for parent_asin in ['B07484MS4G', 'B01FYDF3VI', 'B01C3PMM7S', 'B012HUTKTM', 'B075R6NQ1W', 'B08DCZGQJ5', 'B077WBYPLG', 'B001KYU8UM', 'B0919HCJDN', 'B08MQCXGJM']:
    query = f"""
        SELECT
            parent_asin,
            COUNT(*) AS total_reviews,
            AVG(rating)::NUMERIC(3,2) AS average_rating,
            COUNT(*) FILTER (WHERE rating = 5) AS five_star,
            COUNT(*) FILTER (WHERE rating = 4) AS four_star,
            COUNT(*) FILTER (WHERE rating = 3) AS three_star,
            COUNT(*) FILTER (WHERE rating = 2) AS two_star,
            COUNT(*) FILTER (WHERE rating = 1) AS one_star
        FROM {SCHEMA}.{TABLE_RAW_REVIEWS}
        WHERE parent_asin = %s
        GROUP BY parent_asin
    """
    duration = measure_postgres_query_time(
        conn, query, (parent_asin,)
    )
    durations.append(duration)

print("Product Statistics Query:")
print(f"Median: {np.median(durations):.3f} ms")
print(f"Mean:   {np.mean(durations):.3f} ms")


Product Statistics Query:
Median: 4.369 ms
Mean:   10.429 ms
